# Gene-level Ribosome Profiling from Global BAM + Zarr Count Matrix

This notebook demonstrates how to reconstruct **per-gene, per-sample ribosome footprint profiles** by combining two pipeline outputs:

1. **`unique_reads.bam`** — STAR-aligned BAM of deduplicated unique sequences (one alignment per unique read)
2. **`global_matrix.zarr`** — Zarr count matrix where `counts[read_i, sample_j]` = number of times unique read `i` was observed in sample `j`

### Concept

The pipeline collapses identical reads across all samples into a single FASTA, aligns once, then stores per-sample counts in the zarr matrix. To get a per-sample profile for a gene:

1. **Subset the BAM** — fetch all reads overlapping the gene locus
2. **Map read names → zarr row indices** — read names are `read_{global_id}`
3. **Look up per-sample counts** — inflate each alignment by the count from the zarr matrix
4. **Build position-level coverage** — accumulate counts at each genomic position (or A-site/P-site)

## 0. Configuration

In [ ]:
# === EDIT THESE PATHS ===
BAM_PATH = "/path/to/global/unique_reads.bam"
ZARR_PATH = "/path/to/global/global_matrix.zarr"
GTF_PATH = "/path/to/annotation.gtf"  # Ensembl GTF used for STAR index

# Optional: P-site offset file from RiboWaltz (tab-delimited: length\toffset)
OFFSET_FILE = None  # e.g., "/path/to/psite_offsets.tsv"

# Gene to examine
GENE_NAME = "ACTB"  # or use GENE_ID below
GENE_ID = None       # e.g., "ENSG00000075624" — takes priority over GENE_NAME if set

# Samples to compare (None = all samples)
SAMPLES = None  # e.g., ["SRR1234567", "SRR7654321"]

## 1. Imports & Setup

In [ ]:
import numpy as np
import pysam
import zarr
from pathlib import Path
from collections import defaultdict

try:
    import polars as pl
    HAS_POLARS = True
except ImportError:
    import pandas as pd
    HAS_POLARS = False

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print(f"zarr version: {zarr.__version__}")
print(f"pysam version: {pysam.__version__}")
print(f"Using {'polars' if HAS_POLARS else 'pandas'} for dataframes")

## 2. Load Zarr Matrix

In [ ]:
# Open the zarr store
root = zarr.open(ZARR_PATH, mode='r')
counts = root['counts']

# Read metadata from attrs
n_reads = root.attrs['n_reads']
n_samples = root.attrs['n_samples']
sample_names = list(root.attrs['samples'])
chunk_size = root.attrs['chunk_size']

print(f"Matrix shape: {counts.shape}")
print(f"  Unique reads: {n_reads:,}")
print(f"  Samples:      {n_samples:,}")
print(f"  Chunk size:   {chunk_size:,} reads per chunk")
print(f"\nFirst 10 samples: {sample_names[:10]}")

In [ ]:
# Resolve sample indices
if SAMPLES is not None:
    sample_indices = [sample_names.index(s) for s in SAMPLES]
    selected_samples = SAMPLES
else:
    # Use all samples (for large matrices you may want to subset)
    sample_indices = list(range(n_samples))
    selected_samples = sample_names

print(f"Selected {len(selected_samples)} samples for analysis")

## 3. Parse Gene Coordinates from GTF

In [ ]:
def parse_gtf_attributes(attr_string):
    """Parse GTF attribute string into a dict."""
    attrs = {}
    for field in attr_string.strip().split(';'):
        field = field.strip()
        if not field:
            continue
        key, _, value = field.partition(' ')
        attrs[key] = value.strip('"')
    return attrs


def find_gene_in_gtf(gtf_path, gene_name=None, gene_id=None):
    """
    Find gene boundaries and CDS exons from GTF.
    
    Returns dict with:
      - chrom, start, end, strand (gene-level)
      - cds_intervals: list of (start, end) for CDS exons
      - exon_intervals: list of (start, end) for exons
    """
    gene_info = None
    cds_intervals = []
    exon_intervals = []
    in_gene = False
    target_gene_id = gene_id

    with open(gtf_path) as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split('\t')
            if len(fields) < 9:
                continue

            feature_type = fields[2]
            attrs = parse_gtf_attributes(fields[8])

            # Find the gene
            if feature_type == 'gene':
                if in_gene:
                    break  # We've passed our gene, stop
                
                match = False
                if gene_id and attrs.get('gene_id') == gene_id:
                    match = True
                elif gene_name and attrs.get('gene_name') == gene_name:
                    match = True
                    target_gene_id = attrs.get('gene_id')
                
                if match:
                    gene_info = {
                        'chrom': fields[0],
                        'start': int(fields[3]) - 1,  # GTF is 1-based
                        'end': int(fields[4]),
                        'strand': fields[6],
                        'gene_id': attrs.get('gene_id', ''),
                        'gene_name': attrs.get('gene_name', ''),
                    }
                    in_gene = True
                    continue

            # Collect CDS and exon features for this gene
            if in_gene:
                if attrs.get('gene_id') != target_gene_id:
                    break  # Moved past our gene
                start_0 = int(fields[3]) - 1
                end = int(fields[4])
                if feature_type == 'CDS':
                    cds_intervals.append((start_0, end))
                elif feature_type == 'exon':
                    exon_intervals.append((start_0, end))

    if gene_info is None:
        raise ValueError(f"Gene not found: {gene_name or gene_id}")

    # Deduplicate and sort intervals
    cds_intervals = sorted(set(cds_intervals))
    exon_intervals = sorted(set(exon_intervals))

    gene_info['cds_intervals'] = cds_intervals
    gene_info['exon_intervals'] = exon_intervals
    return gene_info


gene = find_gene_in_gtf(GTF_PATH, gene_name=GENE_NAME, gene_id=GENE_ID)
print(f"Gene: {gene['gene_name']} ({gene['gene_id']})")
print(f"Locus: {gene['chrom']}:{gene['start']}-{gene['end']} ({gene['strand']})")
print(f"CDS exons: {len(gene['cds_intervals'])}")
print(f"Exons: {len(gene['exon_intervals'])}")

## 4. Fetch Reads from BAM Overlapping Gene Locus

In [ ]:
def fetch_gene_reads(bam_path, chrom, start, end, strand=None):
    """
    Fetch all aligned reads overlapping a genomic region.
    
    Returns list of dicts with:
      - global_id: int (row index in zarr matrix)
      - positions: list of aligned reference positions
      - is_reverse: bool
      - aligned_length: int
    """
    reads = []
    bam = pysam.AlignmentFile(bam_path, 'rb')
    
    for read in bam.fetch(chrom, start, end):
        if read.is_unmapped or read.is_secondary or read.is_supplementary:
            continue
        
        # Filter by strand if requested
        if strand is not None:
            if strand == '+' and read.is_reverse:
                continue
            if strand == '-' and not read.is_reverse:
                continue
        
        # Extract global_id from read name: "read_{global_id}"
        name = read.query_name
        if not name.startswith('read_'):
            continue
        global_id = int(name.split('_')[1])
        
        positions = read.get_reference_positions()
        if not positions:
            continue
        
        reads.append({
            'global_id': global_id,
            'positions': positions,
            'is_reverse': read.is_reverse,
            'aligned_length': len(positions),
        })
    
    bam.close()
    return reads


gene_reads = fetch_gene_reads(
    BAM_PATH,
    gene['chrom'],
    gene['start'],
    gene['end'],
    strand=gene['strand'],
)

print(f"Found {len(gene_reads)} unique reads mapping to {gene['gene_name']}")
if gene_reads:
    lengths = [r['aligned_length'] for r in gene_reads]
    print(f"Read length range: {min(lengths)}-{max(lengths)} nt")

## 5. Inflate Counts from Zarr Matrix

For each unique read overlapping the gene, look up its per-sample counts from the zarr matrix. This is efficient because the zarr array is chunked by rows (reads), so reads with nearby `global_id` values share chunks.

In [ ]:
def inflate_read_counts(gene_reads, counts_array, sample_indices):
    """
    Look up zarr counts for each unique read at selected sample columns.
    
    Returns:
        dict: global_id -> np.array of counts (one per selected sample)
    """
    if not gene_reads:
        return {}
    
    global_ids = sorted(set(r['global_id'] for r in gene_reads))
    id_to_counts = {}
    
    # Batch reads by zarr chunk for efficient access
    chunk_size = counts_array.chunks[0]
    by_chunk = defaultdict(list)
    for gid in global_ids:
        by_chunk[gid // chunk_size].append(gid)
    
    for chunk_idx in sorted(by_chunk):
        ids_in_chunk = by_chunk[chunk_idx]
        chunk_start = chunk_idx * chunk_size
        chunk_end = min(chunk_start + chunk_size, counts_array.shape[0])
        
        # Read the full chunk once (all selected sample columns)
        chunk_data = counts_array[chunk_start:chunk_end, :]
        
        for gid in ids_in_chunk:
            local_row = gid - chunk_start
            id_to_counts[gid] = chunk_data[local_row, sample_indices]
    
    return id_to_counts


read_counts = inflate_read_counts(gene_reads, counts, sample_indices)

# Summary stats
total_per_sample = np.zeros(len(selected_samples), dtype=np.uint64)
for gid, c in read_counts.items():
    total_per_sample += c.astype(np.uint64)

print(f"Inflated counts for {len(read_counts)} unique reads across {len(selected_samples)} samples")
print(f"\nTotal reads per sample mapping to {gene['gene_name']}:")
for name, total in zip(selected_samples[:10], total_per_sample[:10]):
    print(f"  {name}: {total:,}")
if len(selected_samples) > 10:
    print(f"  ... ({len(selected_samples) - 10} more)")

## 6. Build Per-Sample Positional Profiles

For each sample, we accumulate coverage at each genomic position by multiplying each unique read's alignment positions by its count in that sample.

If an offset file is provided, we compute the A-site position using length-specific offsets (matching the pipeline's `bam_to_bed.py` logic). Otherwise, we use the 5' end of the read.

In [ ]:
def load_offsets(offset_file):
    """Load P-site/A-site offsets from tab file (length\toffset)."""
    offsets = {}
    with open(offset_file) as f:
        next(f)  # skip header
        for line in f:
            parts = line.strip().split('\t')
            offsets[int(parts[0])] = int(parts[1])
    return offsets


def build_positional_profiles(
    gene_reads, read_counts, sample_indices,
    gene_start, gene_end, offsets=None, mode='coverage'
):
    """
    Build per-sample coverage arrays over the gene locus.
    
    Args:
        gene_reads: list of read dicts from fetch_gene_reads
        read_counts: dict global_id -> np.array of sample counts
        sample_indices: list of sample column indices
        gene_start, gene_end: gene boundaries (0-based)
        offsets: optional dict of length -> offset for A-site calculation
        mode: 'coverage' (full footprint), 'asite' (single position per read)
    
    Returns:
        profiles: np.array of shape (n_samples, gene_length)
        positions: np.array of genomic positions (gene_start..gene_end-1)
    """
    gene_length = gene_end - gene_start
    n_samples = len(sample_indices)
    profiles = np.zeros((n_samples, gene_length), dtype=np.float64)
    positions = np.arange(gene_start, gene_end)
    
    for read in gene_reads:
        gid = read['global_id']
        sample_counts = read_counts.get(gid)
        if sample_counts is None:
            continue
        
        ref_positions = read['positions']
        aligned_len = read['aligned_length']
        
        if mode == 'asite' and offsets is not None:
            offset = offsets.get(aligned_len)
            if offset is None or offset >= aligned_len:
                continue
            # Compute A-site (same logic as bam_to_bed.py)
            if not read['is_reverse']:
                asite = ref_positions[offset]
            else:
                asite = ref_positions[-1 - offset]
            
            idx = asite - gene_start
            if 0 <= idx < gene_length:
                for si in range(n_samples):
                    profiles[si, idx] += sample_counts[si]
        else:
            # Full footprint coverage
            for pos in ref_positions:
                idx = pos - gene_start
                if 0 <= idx < gene_length:
                    for si in range(n_samples):
                        profiles[si, idx] += sample_counts[si]
    
    return profiles, positions


# Load offsets if available
offsets = load_offsets(OFFSET_FILE) if OFFSET_FILE else None
profile_mode = 'asite' if offsets else 'coverage'

profiles, positions = build_positional_profiles(
    gene_reads, read_counts, sample_indices,
    gene['start'], gene['end'],
    offsets=offsets, mode=profile_mode,
)

print(f"Profile mode: {profile_mode}")
print(f"Profile shape: {profiles.shape}  (samples × positions)")
print(f"Genomic range: {gene['chrom']}:{gene['start']}-{gene['end']}")

## 7. Visualise Gene Profiles

In [ ]:
def plot_gene_profiles(
    profiles, positions, sample_names, gene_info,
    max_samples=6, normalise=False,
):
    """
    Plot ribosome footprint profiles across a gene for multiple samples.
    """
    n_to_plot = min(len(sample_names), max_samples)
    fig, axes = plt.subplots(
        n_to_plot, 1,
        figsize=(14, 2.5 * n_to_plot),
        sharex=True,
        squeeze=False,
    )
    
    strand = gene_info['strand']
    chrom = gene_info['chrom']
    
    for i in range(n_to_plot):
        ax = axes[i, 0]
        y = profiles[i]
        
        if normalise and y.sum() > 0:
            y = y / y.sum() * 1e6  # RPM-like
            ylabel = 'Normalised coverage'
        else:
            ylabel = 'Read count'
        
        ax.fill_between(positions, y, alpha=0.7, linewidth=0.5)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(sample_names[i], fontsize=10, loc='left')
        ax.ticklabel_format(axis='x', style='plain')
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
        
        # Shade CDS exons
        for cds_start, cds_end in gene_info['cds_intervals']:
            ax.axvspan(cds_start, cds_end, alpha=0.1, color='green', zorder=0)
    
    axes[-1, 0].set_xlabel(f'{chrom} position ({strand} strand)', fontsize=11)
    fig.suptitle(
        f"{gene_info['gene_name']} ({gene_info['gene_id']})  —  "
        f"{chrom}:{gene_info['start']:,}-{gene_info['end']:,} ({strand})",
        fontsize=13, fontweight='bold', y=1.01,
    )
    plt.tight_layout()
    plt.show()


plot_gene_profiles(
    profiles, positions, selected_samples, gene,
    max_samples=6, normalise=False,
)

In [ ]:
# Normalised version (each sample scaled to 1M total)
plot_gene_profiles(
    profiles, positions, selected_samples, gene,
    max_samples=6, normalise=True,
)

## 8. CDS Metagene Profile

Project the coverage onto CDS-relative coordinates (0 = start codon, normalised to CDS length).

In [ ]:
def cds_relative_profile(profiles, positions, cds_intervals, strand, n_bins=100):
    """
    Map genomic coverage to CDS-relative coordinates.
    
    For + strand: CDS positions are in order start→stop.
    For - strand: CDS positions are reversed.
    
    Returns:
        meta_profiles: np.array (n_samples, n_bins)
        bin_edges: np.array of fractional CDS positions (0-1)
    """
    if not cds_intervals:
        print("No CDS intervals found — skipping metagene")
        return None, None
    
    # Build sorted list of CDS positions
    cds_positions = []
    for cds_start, cds_end in sorted(cds_intervals):
        cds_positions.extend(range(cds_start, cds_end))
    cds_positions = sorted(set(cds_positions))
    
    if strand == '-':
        cds_positions = cds_positions[::-1]
    
    # Map genomic position → CDS-relative index
    pos_to_cds_idx = {pos: i for i, pos in enumerate(cds_positions)}
    cds_length = len(cds_positions)
    
    # Build CDS-relative profiles
    n_samples = profiles.shape[0]
    cds_profiles = np.zeros((n_samples, cds_length), dtype=np.float64)
    
    gene_start = int(positions[0])
    for pos in cds_positions:
        idx = pos - gene_start
        if 0 <= idx < profiles.shape[1]:
            cds_idx = pos_to_cds_idx[pos]
            cds_profiles[:, cds_idx] = profiles[:, idx]
    
    # Bin into n_bins
    bin_edges = np.linspace(0, 1, n_bins + 1)
    meta_profiles = np.zeros((n_samples, n_bins), dtype=np.float64)
    
    for b in range(n_bins):
        start_idx = int(b * cds_length / n_bins)
        end_idx = int((b + 1) * cds_length / n_bins)
        if start_idx < end_idx:
            meta_profiles[:, b] = cds_profiles[:, start_idx:end_idx].mean(axis=1)
    
    return meta_profiles, bin_edges


meta_profiles, bin_edges = cds_relative_profile(
    profiles, positions, gene['cds_intervals'], gene['strand'], n_bins=100,
)

if meta_profiles is not None:
    fig, ax = plt.subplots(figsize=(12, 4))
    x = (bin_edges[:-1] + bin_edges[1:]) / 2 * 100  # % of CDS
    
    # Plot first few samples
    for i in range(min(4, len(selected_samples))):
        y = meta_profiles[i]
        if y.sum() > 0:
            y = y / y.sum() * 100  # percentage
        ax.plot(x, y, label=selected_samples[i], alpha=0.8, linewidth=1.2)
    
    ax.set_xlabel('CDS position (%)', fontsize=12)
    ax.set_ylabel('Relative coverage (%)', fontsize=12)
    ax.set_title(
        f'{gene["gene_name"]} — CDS metagene profile',
        fontsize=13, fontweight='bold',
    )
    ax.legend(fontsize=9, loc='upper right')
    ax.axvline(0, color='green', linestyle='--', alpha=0.4, label='Start codon')
    ax.axvline(100, color='red', linestyle='--', alpha=0.4, label='Stop codon')
    plt.tight_layout()
    plt.show()

## 9. Reading Frame Analysis (A-site mode only)

When P-site/A-site offsets are available, we can check the triplet periodicity within the CDS — a hallmark of active translation.

In [ ]:
def reading_frame_counts(profiles, positions, cds_intervals, strand):
    """
    Count A-site coverage in each reading frame (0, 1, 2) relative to CDS start.
    
    Returns:
        frame_counts: np.array (n_samples, 3)
    """
    if not cds_intervals:
        return None
    
    # Build CDS positions in reading order
    cds_positions = []
    for cds_start, cds_end in sorted(cds_intervals):
        cds_positions.extend(range(cds_start, cds_end))
    cds_positions = sorted(set(cds_positions))
    if strand == '-':
        cds_positions = cds_positions[::-1]
    
    pos_to_frame = {pos: i % 3 for i, pos in enumerate(cds_positions)}
    gene_start = int(positions[0])
    
    n_samples = profiles.shape[0]
    frame_counts = np.zeros((n_samples, 3), dtype=np.float64)
    
    for pos, frame in pos_to_frame.items():
        idx = pos - gene_start
        if 0 <= idx < profiles.shape[1]:
            frame_counts[:, frame] += profiles[:, idx]
    
    return frame_counts


frame_counts = reading_frame_counts(
    profiles, positions, gene['cds_intervals'], gene['strand'],
)

if frame_counts is not None:
    fig, axes = plt.subplots(1, min(4, len(selected_samples)), figsize=(4 * min(4, len(selected_samples)), 3.5))
    if not hasattr(axes, '__len__'):
        axes = [axes]
    
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    for i, ax in enumerate(axes):
        if i >= len(selected_samples):
            break
        fc = frame_counts[i]
        total = fc.sum()
        pcts = fc / total * 100 if total > 0 else fc
        ax.bar([0, 1, 2], pcts, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['Frame 0', 'Frame 1', 'Frame 2'])
        ax.set_ylabel('% of CDS reads')
        ax.set_title(selected_samples[i], fontsize=10)
        ax.set_ylim(0, 100)
    
    fig.suptitle(
        f'{gene["gene_name"]} — Reading Frame Distribution',
        fontsize=13, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()
else:
    print("No CDS intervals — skipping frame analysis")

## 10. Summary Table

In [ ]:
# Build summary dataframe
summary_data = {
    'sample': selected_samples,
    'total_gene_reads': total_per_sample.tolist(),
}

if frame_counts is not None:
    for f in range(3):
        summary_data[f'frame_{f}_pct'] = [
            round(frame_counts[i, f] / frame_counts[i].sum() * 100, 1)
            if frame_counts[i].sum() > 0 else 0.0
            for i in range(len(selected_samples))
        ]

if HAS_POLARS:
    summary_df = pl.DataFrame(summary_data)
else:
    summary_df = pd.DataFrame(summary_data)

print(f"\n{gene['gene_name']} ({gene['gene_id']}) — Per-sample summary:")
print(f"Unique reads overlapping gene: {len(gene_reads)}")
summary_df